In [58]:
# === NOTEBOOK 01: DATA GENERATION v3 (FINAL) ===
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
import random

spark = SparkSession.builder.getOrCreate()

print("="*70)
print(" GENERATE 6.000 UMKM JAWA BARAT - VERSI 3 (SKOR REALISTIS)")
print("="*70)

kabupaten_list = [
    "Bandung", "Bogor", "Bekasi", "Depok", "Tasikmalaya", 
    "Cirebon", "Sukabumi", "Garut", "Ciamis", "Kuningan",
    "Majalengka", "Sumedang", "Indramayu", "Subang", "Purwakarta",
    "Karawang", "Pangandaran", "Banjar", "Cimahi", "Bandung Barat"
]

jenis_usaha_list = ["Makanan", "Fashion", "Kerajinan", "Jasa", "Pertanian"]

schema = StructType([
    StructField("umkm_id", StringType(), True),
    StructField("nama_usaha", StringType(), True),
    StructField("jenis_usaha", StringType(), True),
    StructField("kabupaten", StringType(), True),
    StructField("kecamatan", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("tahun_berdiri", IntegerType(), True),
    StructField("omset_bulanan", IntegerType(), True),
    StructField("jumlah_karyawan", IntegerType(), True),
    StructField("risiko_banjir", DoubleType(), True),
    StructField("persaingan_index", DoubleType(), True),
    StructField("cluster", IntegerType(), True),
    StructField("skor_potensi", DoubleType(), True),
    StructField("infrastructure_score", DoubleType(), True),
    StructField("pop_density_proxy", IntegerType(), True)
])

data = []
for i in range(1, 6001):
    kab = random.choice(kabupaten_list)
    jenis = random.choice(jenis_usaha_list)
    
    base = random.randint(48, 82)
    noise = random.randint(-8, 8)
    skor = float(max(45, min(95, base + noise)))
    
    omset = int(25000000 + (skor - 50) * 4500000 + random.randint(-15000000, 15000000))
    omset = max(8000000, min(380000000, omset))
    
    persaingan = round(max(0.25, min(0.92, 0.85 - (skor - 50) * 0.012 + random.uniform(-0.12, 0.12))), 2)
    risiko = round(max(0.18, min(0.88, 0.78 - (skor - 50) * 0.011 + random.uniform(-0.1, 0.1))), 2)
    
    cluster = random.randint(0, 7)
    infra = float(round(max(48, min(92, 65 + (skor - 55) * 0.6 + random.uniform(-8, 8))), 1))
    pop_density = int(max(1200, min(9200, 3800 + (skor - 50) * 45 + random.randint(-1200, 1200))))
    
    data.append((
        f"JB-2026-{str(i).zfill(5)}",
        f"Usaha {jenis} {i}",
        jenis,
        kab,
        f"Kecamatan {random.randint(1, 42)}",
        round(random.uniform(-7.5, -6.0), 6),
        round(random.uniform(106.5, 108.5), 6),
        random.randint(2015, 2026),
        omset,
        random.randint(1, 32),
        float(risiko),
        float(persaingan),
        cluster,
        skor,
        infra,
        pop_density
    ))

df = spark.createDataFrame(data, schema)

print(f"\n✅ Total data: {df.count()}")
print(f"✅ Skor range: 45 - 95")
print(f"✅ Rata-rata skor: {df.agg({'skor_potensi': 'avg'}).collect()[0][0]:.1f}")

df.write.mode("overwrite").parquet("/lakehouse/default/Files/umkm_v3_final")

print("\n✅ Data berhasil disimpan: /lakehouse/default/Files/umkm_v3_final")
df.select("umkm_id", "kabupaten", "jenis_usaha", "skor_potensi", "cluster").show(8)

StatementMeta(yudhaesparkpool, 2, 65, Finished, Available, Finished, False)

 GENERATE 6.000 UMKM JAWA BARAT - VERSI 3 (SKOR REALISTIS)

✅ Total data: 6000
✅ Skor range: 45 - 95
✅ Rata-rata skor: 65.2

✅ Data berhasil disimpan: /lakehouse/default/Files/umkm_v3_final
+-------------+-------------+-----------+------------+-------+
|      umkm_id|    kabupaten|jenis_usaha|skor_potensi|cluster|
+-------------+-------------+-----------+------------+-------+
|JB-2026-00001|        Garut|    Fashion|        86.0|      2|
|JB-2026-00002|       Banjar|  Kerajinan|        48.0|      3|
|JB-2026-00003|Bandung Barat|  Pertanian|        81.0|      5|
|JB-2026-00004|     Kuningan|    Makanan|        56.0|      2|
|JB-2026-00005|       Banjar|    Makanan|        78.0|      6|
|JB-2026-00006|      Bandung|    Fashion|        71.0|      6|
|JB-2026-00007|    Indramayu|       Jasa|        61.0|      5|
|JB-2026-00008|   Majalengka|  Pertanian|        56.0|      1|
+-------------+-------------+-----------+------------+-------+
only showing top 8 rows

